<a href="https://colab.research.google.com/github/1pawn0/Google-Colab-Public-Notebooks/blob/main/Kaggle-Competitions/contradictory_my_dear_watson_XLM_RoBERTa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
%pip install -qU torch transformers tokenizers kagglehub polars

In [7]:
import os
import shutil
import sys
from pathlib import Path

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import torch
from google.colab import userdata
from transformers import (
    Trainer,
    TrainingArguments,
    XLMRobertaConfig,
    XLMRobertaTokenizer,
    XLMRobertaModel,
    XLMRobertaForCausalLM,
    XLMRobertaForMaskedLM,
    XLMRobertaForSequenceClassification,
)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
import kagglehub

print(
    "python " + sys.version.split()[0],
    "torch " + torch.__version__,
    "polars " + pl.__version__,
    "kagglehub " + kagglehub.__version__,
    sep="\n",
)


python 3.12.11
torch 2.8.0+cu126
polars 1.33.1
kagglehub 0.3.13


### Load the pretrained [`XLM-RoBERTa`](https://huggingface.co/docs/transformers/main/en/model_doc/xlm-roberta) Model

In [8]:
from transformers import (
    Trainer,
    TrainingArguments,
    XLMRobertaConfig,
    XLMRobertaTokenizer,
    XLMRobertaModel,
    XLMRobertaForCausalLM,
    XLMRobertaForMaskedLM,
    XLMRobertaForSequenceClassification,
)

MODEL_NAME = "FacebookAI/xlm-roberta-large"
model_config = XLMRobertaConfig.from_pretrained(MODEL_NAME)
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME) # used for natural language inference (NLI)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Downloading the competition's dataset

In [9]:
competition_name = "contradictory-my-dear-watson"
competition_path: Path = Path(kagglehub.competition.competition_download(competition_name))
competition_files: list = os.listdir(competition_path)
print(competition_files)
competition_data_path: Path = Path(f"./data/{competition_name}")
shutil.copytree(competition_path, competition_data_path, dirs_exist_ok=True)
print(competition_data_path)

['train.csv', 'test.csv', 'sample_submission.csv']
data/contradictory-my-dear-watson


Load all csv files of the competition into polars dataframes

In [10]:
train_df = pl.read_csv(competition_data_path / "train.csv")
test_df = pl.read_csv(competition_data_path / "test.csv")
sample_submission_df = pl.read_csv(competition_data_path / "sample_submission.csv")


Perform EDA on the loaded dataframes